# Feature Engineering — Model-Ready Dataset\n\nThis notebook engineers all features defined in Phase 2 and validates the output.\nThe logic here is prototyped inline, then refactored into `src/features/engineer.py` for reuse.

In [1]:
import os, sys, warnings
from pathlib import Path

import pandas as pd
import numpy as np
import holidays
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

def get_engine():
    user = os.getenv("DB_USER")
    pw   = os.getenv("DB_PASSWORD", "")
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    name = os.getenv("DB_NAME")
    return create_engine(f"postgresql+psycopg2://{user}:{pw}@{host}:{port}/{name}")

engine = get_engine()

## 1. Load and merge raw data\n\nWe join `spot_prices`, `generation`, and `weather` into a single DataFrame keyed on `(timestamp_utc, bidding_zone)`.\n\nWeather is location-based (Aarhus, Copenhagen), so we pivot it to wide format first — one column per location per variable. The merge is a **left join** from prices to generation and weather so we keep every price row even if generation/weather has gaps.

In [2]:
# Load raw tables
prices = pd.read_sql("SELECT timestamp_utc, bidding_zone, price_eur_mwh FROM spot_prices ORDER BY timestamp_utc",
                     engine, parse_dates=["timestamp_utc"])
gen = pd.read_sql("SELECT timestamp_utc, bidding_zone, wind_onshore_mw, wind_offshore_mw, solar_mw FROM generation ORDER BY timestamp_utc",
                  engine, parse_dates=["timestamp_utc"])
weather = pd.read_sql("SELECT timestamp_utc, location, temperature_c, wind_speed_ms FROM weather ORDER BY timestamp_utc",
                      engine, parse_dates=["timestamp_utc"])

# Pivot weather to wide format
weather_aarhus = weather[weather.location == "Aarhus"][["timestamp_utc", "temperature_c", "wind_speed_ms"]].rename(
    columns={"temperature_c": "temp_aarhus", "wind_speed_ms": "wind_speed_aarhus"})
weather_cph = weather[weather.location == "Copenhagen"][["timestamp_utc", "temperature_c", "wind_speed_ms"]].rename(
    columns={"temperature_c": "temp_cph", "wind_speed_ms": "wind_speed_cph"})
weather_wide = weather_aarhus.merge(weather_cph, on="timestamp_utc", how="outer")

# Merge: prices ← generation ← weather
df = prices.merge(gen, on=["timestamp_utc", "bidding_zone"], how="left")
df = df.merge(weather_wide, on="timestamp_utc", how="left")
df = df.sort_values(["bidding_zone", "timestamp_utc"]).reset_index(drop=True)

print(f"Merged shape: {df.shape}")
df.head()

Merged shape: (18276, 10)


,timestamp_utc,bidding_zone,price_eur_mwh,wind_onshore_mw,wind_offshore_mw,solar_mw,temp_aarhus,wind_speed_aarhus,temp_cph,wind_speed_cph
0,2022-02-01 00:00:00,DK1,138.99,1237.96,1062.17,3.23,1.8,6.45,-1.0,2.71
1,2022-02-01 01:00:00,DK1,135.51,1350.11,1067.68,3.25,1.1,5.85,-0.6,3.32
2,2022-02-01 02:00:00,DK1,130.83,1378.46,1100.43,3.26,1.1,6.14,0.5,3.89
3,2022-02-01 03:00:00,DK1,137.54,1224.80,1107.97,3.24,0.8,5.61,1.1,5.51
4,2022-02-01 04:00:00,DK1,143.44,939.07,1047.77,3.24,0.8,5.38,1.1,5.20


## 2. Lag Features\n\nPrice lags capture the autoregressive structure identified in the EDA (strong ACF at lags 1, 24, 48, 168). Rolling statistics smooth out hourly noise and represent the recent price regime.\n\n**Important:** Lags are computed *per bidding zone* to avoid data leakage across zones.

In [3]:
# --- Lag features (per zone) ---
lag_configs = [
    ("price_lag_1h",   1),
    ("price_lag_2h",   2),
    ("price_lag_24h",  24),
    ("price_lag_48h",  48),
    ("price_lag_168h", 168),
]

for col_name, lag in lag_configs:
    df[col_name] = df.groupby("bidding_zone")["price_eur_mwh"].shift(lag)

# Rolling features (per zone)
for zone in df.bidding_zone.unique():
    mask = df.bidding_zone == zone
    df.loc[mask, "price_rolling_mean_24h"]  = df.loc[mask, "price_eur_mwh"].rolling(24, min_periods=1).mean()
    df.loc[mask, "price_rolling_std_24h"]   = df.loc[mask, "price_eur_mwh"].rolling(24, min_periods=1).std()
    df.loc[mask, "price_rolling_mean_168h"] = df.loc[mask, "price_eur_mwh"].rolling(168, min_periods=1).mean()

print("Lag features added:", [c for c in df.columns if "lag" in c or "rolling" in c])

Lag features added: ['price_lag_1h', 'price_lag_2h', 'price_lag_24h', 'price_lag_48h', 'price_lag_168h', 'price_rolling_mean_24h', 'price_rolling_std_24h', 'price_rolling_mean_168h']


## 3. Calendar Features\n\nElectricity demand follows strong calendar patterns: weekday vs weekend, hour of day, and season.\n\n**Why sin/cos encoding?** Raw integers (hour 0-23) mislead the model: it treats hour 23 and hour 0 as maximally distant, when they are actually adjacent. Cyclical sin/cos encoding places them correctly on a circle:\n- `hour_sin = sin(2 * pi * hour / 24)`\n- `hour_cos = cos(2 * pi * hour / 24)`\n\nSame logic applies to month (1-12).

In [4]:
# --- Calendar features ---
df["hour"]        = df.timestamp_utc.dt.hour
df["hour_sin"]    = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"]    = np.cos(2 * np.pi * df["hour"] / 24)
df["day_of_week"] = df.timestamp_utc.dt.dayofweek         # 0=Mon
df["is_weekend"]  = df["day_of_week"].isin([5, 6])
df["month"]       = df.timestamp_utc.dt.month
df["month_sin"]   = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"]   = np.cos(2 * np.pi * df["month"] / 12)

# Danish public holidays
dk_holidays = holidays.Denmark(years=range(2022, 2025))
df["is_danish_holiday"] = df.timestamp_utc.dt.date.map(lambda d: d in dk_holidays)

print("Calendar features added. Holiday count:", df.is_danish_holiday.sum())

Calendar features added. Holiday count: 386


## 4. Generation Features\n\nWind and solar generation directly affect the merit-order clearing price. We combine onshore + offshore into a single `wind_total_mw` and compute a `renewables_ratio` as a proxy for the share of zero-marginal-cost supply.

In [5]:
# --- Generation features ---
df["wind_total_mw"] = df["wind_onshore_mw"].fillna(0) + df["wind_offshore_mw"].fillna(0)

# Generation lags (per zone)
df["wind_lag_1h"]  = df.groupby("bidding_zone")["wind_total_mw"].shift(1)
df["wind_lag_24h"] = df.groupby("bidding_zone")["wind_total_mw"].shift(24)

# solar_mw is already present from merge
# Renewables ratio: (wind + solar) / (wind + solar + small epsilon to avoid div-by-zero)
total_ren = df["wind_total_mw"] + df["solar_mw"].fillna(0)
df["renewables_ratio"] = total_ren / (total_ren + 1e-6)  # bounded [0, ~1]

print("Generation features added.")

Generation features added.


## 5. Weather Features\n\nTemperature and wind speed are already in the merged DataFrame from the weather join. We add a `temp_mean_dk` as a simple national average.

In [6]:
# --- Weather features ---
df["temp_mean_dk"] = (df["temp_aarhus"].fillna(0) + df["temp_cph"].fillna(0)) / 2

print("Weather features: temp_aarhus, temp_cph, wind_speed_aarhus, wind_speed_cph, temp_mean_dk")

Weather features: temp_aarhus, temp_cph, wind_speed_aarhus, wind_speed_cph, temp_mean_dk


## 6. Target Variable\n\n`price_next_24h` = the price 24 hours from now. This is our forecast target — predicting what the day-ahead price will be at the same hour tomorrow. We shift the price column back by 24 rows (per zone) to create it.

In [7]:
# --- Target: price 24 hours ahead ---
df["price_next_24h"] = df.groupby("bidding_zone")["price_eur_mwh"].shift(-24)

print(f"Target null count (expected ~48 per zone — last 24h have no target): {df.price_next_24h.isna().sum()}")

Target null count (expected ~48 per zone — last 24h have no target): 48


## 7. Quality Checks & Cleanup

In [8]:
# --- Quality checks ---
print("=== Quality Checks ===\n")

# Check 1: rows where ALL lag features are null (first 168 rows per zone — expected)
lag_cols = [c for c in df.columns if "lag" in c and "price" in c]
all_lag_null = df[lag_cols].isna().all(axis=1).sum()
print(f"1. Rows with ALL price-lag features null: {all_lag_null}")

# Check 2: target nulls
target_null = df.price_next_24h.isna().sum()
print(f"2. Rows with null target (price_next_24h):  {target_null}")

# Check 3: rolling feature outlier check
for col in ["price_rolling_mean_24h", "price_rolling_std_24h", "price_rolling_mean_168h"]:
    rng = df[col].max() - df[col].min()
    print(f"3. {col}: min={df[col].min():.1f}  max={df[col].max():.1f}  range={rng:.1f}")

# Drop rows where target is null (can't train on them)
rows_before = len(df)
df = df.dropna(subset=["price_next_24h"]).reset_index(drop=True)
rows_after = len(df)
print(f"\nDropped {rows_before - rows_after} rows with null target.")
print(f"Final dataset: {rows_after:,} rows x {len(df.columns)} columns")

=== Quality Checks ===

1. Rows with ALL price-lag features null: 2
2. Rows with null target (price_next_24h):  48
3. price_rolling_mean_24h: min=-61.4  max=708.7  range=770.2
3. price_rolling_std_24h: min=0.5  max=287.1  range=286.6
3. price_rolling_mean_168h: min=16.1  max=608.1  range=592.0

Dropped 48 rows with null target.
Final dataset: 18,228 rows x 33 columns


In [9]:
# --- Final summary ---
print("=== Feature Summary ===\n")
print(f"Rows:          {len(df):,}")
print(f"Columns:       {len(df.columns)}")
print(f"Bidding zones: {df.bidding_zone.unique().tolist()}")
print(f"Date range:    {df.timestamp_utc.min().date()} → {df.timestamp_utc.max().date()}")
print(f"\nNull counts per column:")
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string() if (nulls > 0).any() else "  No nulls!")\n
print(f"\nAll columns: {list(df.columns)}")

SyntaxError: unexpected character after line continuation character (1191615767.py, line 9)